# Lesson 02 — State in Depth

## Objective
Understand how information is stored, accessed, and updated in LangGraph.

## Concepts
- TypedDict
- Initial State
- Partial State Updates
- State Merging
- Multiple State Fields

In [10]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

print("Imports successful!")

Imports successful!


In [11]:
class AgentState(TypedDict):
    query: str
    percentage: float
    number: float
    result: float

In [12]:
def analyze_node(state: AgentState):

    print("Analyzing query:", state["query"])

    return {
        "percentage": 20.0,
        "number": 5000.0
    }

In [13]:
def calculator_node(state: AgentState):

    percentage = state["percentage"]
    number = state["number"]

    result = number * percentage / 100

    print(f"Calculating {percentage}% of {number}")

    return {
        "result": result
    }

In [14]:
builder = StateGraph(AgentState)

builder.add_node("analyze", analyze_node)
builder.add_node("calculator", calculator_node)

builder.add_edge(START, "analyze")
builder.add_edge("analyze", "calculator")
builder.add_edge("calculator", END)

graph = builder.compile()

print("State workflow compiled!")

State workflow compiled!


In [15]:
initial_state = {
    "query": "Calculate 20% of 5000",
    "percentage": 0.0,
    "number": 0.0,
    "result": 0.0
}

final_state = graph.invoke(initial_state)

print("\nFinal State:")
print(final_state)

Analyzing query: Calculate 20% of 5000
Calculating 20.0% of 5000.0

Final State:
{'query': 'Calculate 20% of 5000', 'percentage': 20.0, 'number': 5000.0, 'result': 1000.0}


In [16]:
for step in graph.stream(initial_state, stream_mode="updates"):
    print(step)

Analyzing query: Calculate 20% of 5000
{'analyze': {'percentage': 20.0, 'number': 5000.0}}
Calculating 20.0% of 5000.0
{'calculator': {'result': 1000.0}}
